In [ ]:
import pandas as pd
from typing import Any
import requests
import pandas as pd
import matplotlib.pyplot as plt
from time import sleep
import os
from dotenv import load_dotenv
import psycopg

In [132]:
load_dotenv()
API_KEY=os.environ.get("API_KEY")
os.environ.get("POSTGRESQL_URL_KEY")
#API_KEY = os.getenv("API_KEY")
FECHA_INICIO = "2026-02-1T00:00:00UTC"
fecha_str=FECHA_INICIO
fecha=pd.to_datetime(fecha_str)
fecha_15_dias_atras = fecha + pd.Timedelta(days=15)
FECHA_FIN = fecha_15_dias_atras.strftime("%Y-%m-%dT%H:%M:%SUTC")
print(f"FECHA_INICIO: {FECHA_INICIO}")
print(f"FECHA_FIN: {FECHA_FIN}")
print(f"Diferencia de días: {pd.to_datetime(FECHA_FIN)-pd.to_datetime(FECHA_INICIO)}")



FECHA_INICIO: 2026-02-1T00:00:00UTC
FECHA_FIN: 2026-02-16T00:00:00UTC
Diferencia de días: 15 days 00:00:00


In [4]:
#Rango de fechas no puede ser superior a 15 días.

def datos_todas_estaciones(api_key:str , FECHA_INICIO:str, FECHA_FIN:str) -> tuple[pd.DataFrame, pd.DataFrame, Any]:
    url = (
    "https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/"
    f"fechaini/{FECHA_INICIO}/"
    f"fechafin/{FECHA_FIN}/"
    "todasestaciones"
    )

    respuesta = requests.get(url,params={"api_key": api_key},timeout=30)
    respuesta.raise_for_status()
    resultado = respuesta.json()

    if resultado.get("estado") != 200:
        raise RuntimeError(
            f"Error de AEMET: {resultado.get('descripcion')}"
        )

    respuesta_datos = requests.get(resultado["datos"],timeout=30)
    respuesta_metadatos = requests.get(resultado["metadatos"],timeout=30)
    respuesta_datos.raise_for_status()
    respuesta_metadatos.raise_for_status()

    datos = respuesta_datos.json()
    metadatos = respuesta_metadatos.json()
    df_datos = pd.DataFrame(datos)
    df_metadatos =pd.DataFrame(metadatos)
    
    return df_datos, df_metadatos, resultado.get("estado")





In [20]:
pd.set_option("display.max_columns", None)
datos=datos_todas_estaciones(api_key=os.environ.get("API_KEY"),FECHA_INICIO=FECHA_INICIO,FECHA_FIN=FECHA_FIN)[0]
metadatos=datos_todas_estaciones(api_key=os.environ.get("API_KEY"),FECHA_INICIO=FECHA_INICIO,FECHA_FIN=FECHA_FIN)[1]
df_campos = pd.DataFrame(metadatos["campos"].tolist()).set_index("id")[["descripcion", "tipo_datos", "requerido"]].T


In [21]:
df_campos

id,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,dir,velmedia,racha,horaracha,sol,presmax,horapresmax,presmin,horapresmin,hrmedia,hrmax,horahrmax,hrmin,horahrmin,pintmax,horapintmax
descripcion,fecha del dia (AAAA-MM-DD),indicativo climatolÃ³gico,nombre (ubicaciÃ³n) de la estaciÃ³n,provincia de la estaciÃ³n,altitud de la estaciÃ³n en m sobre el nivel de...,Temperatura media diaria,PrecipitaciÃ³n diaria de 07 a 07,Temperatura MÃ­nima del dÃ­a,Hora y minuto de la temperatura mÃ­nima,Temperatura MÃ¡xima del dÃ­a,Hora y minuto de la temperatura mÃ¡xima,DirecciÃ³n de la racha mÃ¡xima,Velocidad media del viento,Racha mÃ¡xima del viento,Hora y minuto de la racha mÃ¡xima,InsolaciÃ³n,PresiÃ³n mÃ¡xima al nivel de referencia de la ...,Hora de la presiÃ³n mÃ¡xima (redondeada a la h...,PresiÃ³n mÃ­nima al nivel de referencia de la ...,Hora de la presiÃ³n mÃ­nima (redondeada a la h...,Humedad relativa media diaria,Humedad relativa mÃ¡xima diaria,Hora de la humedad relativa mÃ¡xima diaria,Humedad relativa mÃ­nima diaria,Hora de la humedad relativa mÃ­nima diaria,Intensidad mÃ¡xima de precipitaciÃ³n,Hora de la intensidad mÃ¡xima de precipitaciÃ³n
tipo_datos,string,string,string,string,float,float,float,float,string,float,string,float,float,float,string,float,float,string,float,string,float,float,string,float,string,float,string
requerido,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
# i=0
# FECHA_INICIO="2024-01-1T00:00:00UTC"
# fecha_inicial=FECHA_INICIO
# df_datos= pd.DataFrame()
# while i <26:
#     fecha_ini_pd=pd.to_datetime(fecha_inicial)
#     fecha_ini_xi = fecha_ini_pd + pd.Timedelta(days=(15*i))
#     fecha_fin_xi = fecha_ini_pd + pd.Timedelta(days=(15+(15*i)))

#     fecha_ini_xi_str = fecha_ini_xi.strftime("%Y-%m-%dT%H:%M:%SUTC")
#     fecha_din_xi_str = fecha_fin_xi.strftime("%Y-%m-%dT%H:%M:%SUTC")
    
#     respuesta=datos_todas_estaciones(os.environ.get("API_KEY"),fecha_ini_xi_str,fecha_din_xi_str)
#     df_datos= pd.concat([df_datos, respuesta[0]],ignore_index=True)
#     estado=respuesta[2]
#     print(f"Respuesta AEMET: {estado}. Datos guardados desde {fecha_ini_xi_str[:-12]} a {fecha_din_xi_str[:-12]}")
#     i=i+1
#     sleep(15)




In [ ]:
# BUCLE PARA OBTENER HISTORICO DE AEMET DE TODAS LAS ESTACIONES POR AÑOS (SEMANAS, mínimo 2 semanas)
i=0
#hoy= pd.Timestamp.now().normalize()
#FECHA_INICIO="2024-01-1T00:00:00UTC"
años=2/52
fecha_inicial_raw=pd.Timestamp.now()-pd.Timedelta(weeks=52*años)
fecha_inicial=fecha_inicial_raw.normalize().strftime("%Y-%m-%dT%H:%M:%SUTC")
df_datos= pd.DataFrame()


while i <52*años/2:
    fecha_ini_pd=pd.to_datetime(fecha_inicial)
    fecha_ini_xi = fecha_ini_pd + pd.Timedelta(weeks=(i*2))
    fecha_fin_xi = fecha_ini_pd + pd.Timedelta(weeks=(i*2+2))-pd.Timedelta(days=(1))
    

    fecha_ini_xi_str = fecha_ini_xi.strftime("%Y-%m-%dT%H:%M:%SUTC")
    fecha_din_xi_str = fecha_fin_xi.strftime("%Y-%m-%dT%H:%M:%SUTC")
   
    
    respuesta=datos_todas_estaciones(os.environ.get("API_KEY"),fecha_ini_xi_str,fecha_din_xi_str)
    df_datos= pd.concat([df_datos, respuesta[0]],ignore_index=True)
    estado=respuesta[2]
    print(f"[{i+1}de{52*años/2}] Respuesta AEMET: {estado}. Datos guardados del {fecha_ini_xi_str[:-12]} al {fecha_din_xi_str[:-12]}")
    i=i+1
    sleep(3)
to_transform:bool=True

[1de1.0] Respuesta AEMET: 200. Datos guardados del 2026-07-15 al 2026-07-28


In [ ]:
df_datos.head(50)

In [ ]:
#LIMPIEZA DE COLUMNAS 

columnas_enteras = [
    "altitud",
    "hrMedia",
    "hrMax",
    "hrMin",
    "dir",
    "horaPresMax",
    "horaPresMin"
]
columnas_float =[
    "tmed",
    "prec",
    "tmin",
    "tmax",
    "pintMax",
    "velmedia",
    "racha",
    "presMax",
    "presMin",
    "sol"
]
columnas_hora=[
    "horatmin",
    "horatmax",
    "horaHrMax",
    "horaHrMin",
    "horaracha",
    "horaPIntMax"
]



if to_transform:
    df_datos["fecha"] = pd.to_datetime(df_datos["fecha"],format="%Y-%m-%d")
    df_datos[columnas_enteras] = (df_datos[columnas_enteras].apply(pd.to_numeric, errors="coerce").astype("Int64"))
    df_datos[columnas_float] = (df_datos[columnas_float].apply(lambda columna: pd.to_numeric(columna.astype("string").str.strip().str.replace(",", ".", regex=False),errors="coerce")).astype("Float64"))
    df_datos[columnas_hora] = df_datos[columnas_hora].apply(lambda columna: pd.to_numeric(columna.str.strip().str.split(":").str[0],errors="coerce").astype("Int64"))
    to_transform=False




In [121]:
df_datos.to_pickle("ALL_10_YEARS")

In [ ]:
df_datos.head(50)

In [ ]:

df_campos

id,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,dir,velmedia,racha,horaracha,sol,presmax,horapresmax,presmin,horapresmin,hrmedia,hrmax,horahrmax,hrmin,horahrmin,pintmax,horapintmax
descripcion,fecha del dia (AAAA-MM-DD),indicativo climatolÃ³gico,nombre (ubicaciÃ³n) de la estaciÃ³n,provincia de la estaciÃ³n,altitud de la estaciÃ³n en m sobre el nivel de...,Temperatura media diaria,PrecipitaciÃ³n diaria de 07 a 07,Temperatura MÃ­nima del dÃ­a,Hora y minuto de la temperatura mÃ­nima,Temperatura MÃ¡xima del dÃ­a,Hora y minuto de la temperatura mÃ¡xima,DirecciÃ³n de la racha mÃ¡xima,Velocidad media del viento,Racha mÃ¡xima del viento,Hora y minuto de la racha mÃ¡xima,InsolaciÃ³n,PresiÃ³n mÃ¡xima al nivel de referencia de la ...,Hora de la presiÃ³n mÃ¡xima (redondeada a la h...,PresiÃ³n mÃ­nima al nivel de referencia de la ...,Hora de la presiÃ³n mÃ­nima (redondeada a la h...,Humedad relativa media diaria,Humedad relativa mÃ¡xima diaria,Hora de la humedad relativa mÃ¡xima diaria,Humedad relativa mÃ­nima diaria,Hora de la humedad relativa mÃ­nima diaria,Intensidad mÃ¡xima de precipitaciÃ³n,Hora de la intensidad mÃ¡xima de precipitaciÃ³n
tipo_datos,string,string,string,string,float,float,float,float,string,float,string,float,float,float,string,float,float,string,float,string,float,float,string,float,string,float,string
requerido,True,True,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
urls_respuesta=respuesta.json()
url_datos=urls_respuesta["datos"]
url_metadatos=urls_respuesta["metadatos"]

respuesta_datos=requests.get(url_datos,timeout=30)
respuesta_datos.raise_for_status()

estaciones = respuesta_datos.json()

df_estaciones = pd.DataFrame(estaciones)
df_estaciones[df_estaciones["nombre"].str.contains("MADRID",case=False,na=False)]


In [137]:
#Obtención de las carateristicas de cada estación

url = (
    "https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/"
    "todasestaciones"
)

respuesta = requests.get(url,params={"api_key": API_KEY},timeout=30)
respuesta.raise_for_status()
resultado = respuesta.json()

if resultado.get("estado") != 200:
    raise RuntimeError(
        f"Error de AEMET: {resultado.get('descripcion')}"
    )

respuesta_datos = requests.get(
    resultado["datos"],
    timeout=30
)

respuesta_metadatos = requests.get(
    resultado["metadatos"],
    timeout=30
)

print(respuesta_datos.raise_for_status())
print(respuesta_metadatos.raise_for_status())

datos = respuesta_datos.json()
metadatos = respuesta_metadatos.json()
df_datos_info_estaciones = pd.DataFrame(datos)
df_info_estaciones_metadatos =pd.DataFrame(metadatos)
df_info_estaciones_metadatos=pd.DataFrame(df_info_estaciones_metadatos["campos"].tolist()).set_index("id")[["descripcion", "tipo_datos", "requerido"]].T

None
None


In [134]:
df_datos_info_estaciones

,latitud,provincia,altitud,indicativo,nombre,indsinop,longitud
0,394924N,ILLES BALEARS,490,B013X,"ESCORCA, LLUC",08304,025309E
1,394744N,BALEARES,5,B051A,"SÓLLER, PUERTO",08316,024129E
2,394121N,ILLES BALEARS,60,B087X,BANYALBUFAR,,023046E
3,393446N,BALEARES,52,B103B,ANDRATX - SANT ELM,,022208E
4,393305N,BALEARES,50,B158X,"CALVIÀ, ES CAPDELLÀ",,022759E
...,...,...,...,...,...,...,...
915,424131N,LLEIDA,2467,9988B,CAP DE VAQUÈIRA,08936,005826E
916,424201N,LLEIDA,1161,9990X,"NAUT ARAN, ARTIES",08107,005237E
917,424634N,LLEIDA,722,9994X,BOSSÒST,,004123E
918,430528N,NAVARRA,334,9995Y,VALCARLOS/LUZAIDE,,011803W


In [42]:
df=pd.read_pickle("ALL_10_YEARS")

In [ ]:
df

In [ ]:
df=pd.read_pickle("ALL_10_YEARS")
columnas_enteras = [
    "altitud",
    "hrMedia",
    "hrMax",
    "hrMin",
    "dir" ,
    "horaPresMax",
    "horaPresMin" 
]
columnas_float =[
    "tmed",
    "prec",
    "tmin",
    "tmax",
    "pintMax",
    "velmedia",
    "racha",
    "presMax",
    "presMin",
    "sol"
]
columnas_hora=[
    "horatmin",
    "horatmax",
    "horaHrMax",
    "horaHrMin",
    "horaracha",
    
    "horaPIntMax"
]

columnas_horas_int=[
    "horaPresMax",
    "horaPresMin"
]



df["fecha"] = pd.to_datetime(df["fecha"],format="%Y-%m-%d")
df[columnas_enteras] = (df[columnas_enteras].apply(pd.to_numeric, errors="coerce").astype("Int64"))
df[columnas_float] = (df[columnas_float].apply(lambda columna: pd.to_numeric(columna.astype("string").str.strip().str.replace(",", ".", regex=False),errors="coerce")).astype("Float64"))

for columna in columnas_hora:
    df[f"{columna}_son_varias_h"] = (df[columna].eq("Varias"))
for columna in columnas_hora:
    df[columna] = (pd.to_datetime(df[columna].astype("string").str.strip(),format="%H:%M",errors="coerce").dt.strftime("%H:%M"))
    #.dt.strftime("%H:%M")
for columna in columnas_horas_int:
    df[columna] =df[columna].replace(24, 0).astype("Int64").astype("string") + ":00"
    df[columna] = (pd.to_datetime(df[columna].astype("string").str.strip(),format="%H:%M",errors="coerce").dt.strftime("%H:%M"))

df_limpio = df.astype(object).where(pd.notna(df), None)



In [ ]:
df

In [ ]:

# Convertimos los NaN/NaT de pandas en None para PostgreSQL
# df_limpio = df.astype(object).where(pd.notna(df), None)
df_limpio=df_limpio.head(50)
#df_limpio.loc[0,"provincia"]="TEST REMPLAZO"
conn_str:str ="postgresql://" + os.environ.get("POSTGRESQL_URL_KEY")


with psycopg.connect(conn_str) as conn:
    with conn.cursor() as cur:

        # Crear la tabla si todavía no existe
        cur.execute("""
            CREATE TABLE IF NOT EXISTS all_stations_measurements (
            fecha DATE,
	        indicativo CHAR (5),
	        nombre VARCHAR,
	        provincia VARCHAR,
	        altitud INTEGER,
	        tmed FLOAT,
	        prec FLOAT,
	        tmin FLOAT,
	        horatmin TIME,
	        tmax FLOAT,
	        horatmax TIME,
	        hrMedia INTEGER,
	        hrMax INTEGER,
	        horaHrMax TEXT,
	        hrMin INTEGER,
	        horaHrMin TEXT,
	        pintMax FLOAT,
	        dir INTEGER,
	        velmedia FLOAT,	
	        racha FLOAT,
	        horaracha TEXT,
	        presMax	FLOAT,
	        horaPresMax	TEXT,
	        presMin	FLOAT,
	        horaPresMin	TEXT,
	        sol	FLOAT,
	        horaPIntMax TEXT,
	        horatmin_son_varias_h BOOL,
	        horatmax_son_varias_h BOOL,
	        horaHrMax_son_varias_h BOOL,
	        horaHrMin_son_varias_h BOOL,
    	    horaracha_son_varias_h BOOL,
	        horaPIntMax_son_varias_h BOOL,
            
            CONSTRAINT fecha_indicativo_unique
                UNIQUE (fecha, indicativo)

            );
        """)

        # Insertar todas las filas del DataFrame
        cur.executemany(
            """
            INSERT INTO all_stations_measurements(
                fecha,
                indicativo,
                nombre,
                provincia,
                altitud,
                tmed,
                prec,
                tmin,
                horatmin,
                tmax,
                horatmax,
                hrMedia,
                hrMax,
                horaHrMax,
                hrMin,
                horaHrMin,
                pintMax,
                dir,
                velmedia,
                racha,
                horaracha,
                presMax,
                horaPresMax,
                presMin,
                horaPresMin,
                sol,
                horaPIntMax,
                horatmin_son_varias_h,
                horatmax_son_varias_h,
                horaHrMax_son_varias_h,
                horaHrMin_son_varias_h,
                horaracha_son_varias_h,
                horaPIntMax_son_varias_h        
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (fecha, indicativo)
            DO UPDATE SET
            nombre = EXCLUDED.nombre, 
            provincia = EXCLUDED.provincia,
            altitud = EXCLUDED.altitud,
            tmed = EXCLUDED.tmed,
            prec = EXCLUDED.prec,
            tmin = EXCLUDED.tmin,
            horatmin = EXCLUDED.horatmin,
            tmax = EXCLUDED.tmax,
            horatmax = EXCLUDED.horatmax,
            hrMedia = EXCLUDED.hrMedia,
            hrMax = EXCLUDED.hrMax,
            horaHrMax =  EXCLUDED.horaHrMax,
            hrMin = EXCLUDED.hrMin,
            horaHrMin = EXCLUDED.horaHrMin,
            pintMax = EXCLUDED.pintMax,
            dir = EXCLUDED.dir,
            velmedia = EXCLUDED.velmedia,
            racha = EXCLUDED.racha,
            horaracha = EXCLUDED.horaracha,
            presMax = EXCLUDED.presMax,
            horaPresMax = EXCLUDED.horaPresMax,
            presMin = EXCLUDED.presMin,
            horaPresMin = EXCLUDED.horaPresMin,
            sol = EXCLUDED.sol,
            horaPIntMax = EXCLUDED.horaPIntMax,
            horatmin_son_varias_h = EXCLUDED.horatmin_son_varias_h,
            horatmax_son_varias_h = EXCLUDED.horatmax_son_varias_h,
            horaHrMax_son_varias_h = EXCLUDED.horaHrMax_son_varias_h,
            horaHrMin_son_varias_h = EXCLUDED.horaHrMin_son_varias_h,
            horaracha_son_varias_h = EXCLUDED.horaracha_son_varias_h,
            horaPIntMax_son_varias_h = EXCLUDED.horaPIntMax_son_varias_h;

            """,
            df_limpio.itertuples(index=False, name=None)
        )

    # No es estrictamente necesario dentro del with,
    # pero puedes dejarlo explícito
    conn.commit()